# Análisis de Cancelación de Clientes

En este cuaderno reproducimos un análisis simplificado del problema de *churn* o cancelación de clientes en una compañía de telecomunicaciones. La estructura sigue los pasos de limpieza, exploración de datos, modelado con Regresión Logística y Random Forest, y evaluación de métricas clave.

> **Nota:** El conjunto de datos original (`WA_Fn-UseC_-Telco-Customer-Churn.csv`) puede descargarse desde Kaggle u otras fuentes públicas. Aquí empleamos un conjunto de datos sintético generado con `sklearn.datasets.make_classification` para ilustrar el flujo de trabajo; al sustituir el generador por la carga del CSV original se obtendrán resultados similares.


In [ ]:
# Importación de librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
%matplotlib inline

## 1. Carga de datos y síntesis de conjunto

Si se dispone del archivo CSV original, se puede cargar con:

```python
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
```

En este ejemplo generaremos un conjunto de datos sintético para ilustrar los pasos subsiguientes.


In [ ]:
# Generación de conjunto de datos sintético con una proporción de churn similar (≈26.5 %)
X, y = make_classification(n_samples=7032,
                           n_features=20,
                           n_informative=10,
                           n_redundant=5,
                           n_clusters_per_class=2,
                           weights=[0.735, 0.265],
                           class_sep=0.55,
                           flip_y=0.07,
                           random_state=42)

# Convertimos a DataFrame para tener nombres de variables (feature_0 ... feature_19)
columns = [f'feature_{i}' for i in range(X.shape[1])]
df = pd.DataFrame(X, columns=columns)
df['Churn'] = y
df.head()

## 2. Exploración del conjunto de datos

Comprobamos la distribución de la variable objetivo y visualizamos un gráfico de barras.


In [ ]:
# Distribución de la variable de salida
churn_counts = df['Churn'].value_counts().rename(index={0:'No Churn', 1:'Churn'})
churn_counts

# Gráfico de barras
sns.set_style('whitegrid')
plt.figure(figsize=(5,3))
sns.barplot(x=churn_counts.index, y=churn_counts.values, palette='Blues')
plt.title('Distribución de Churn')
plt.ylabel('Número de clientes')
plt.show()

## 3. Preparación de los datos

Dividimos el conjunto en variables de entrada (*X*) y salida (*y*), normalizamos mediante `StandardScaler` (solo para la Regresión Logística) y reservamos un conjunto de prueba.


In [ ]:
# División en variables de entrada y salida
X = df.drop('Churn', axis=1)
y = df['Churn']

# Conjunto de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Estandarización para la Regresión Logística
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 4. Entrenamiento de modelos

Entrenamos dos algoritmos diferentes: Regresión Logística y Random Forest. Para este último limitamos la profundidad del árbol y el número de estimadores para obtener un balance entre interpretación y precisión.


In [ ]:
# Regresión Logística
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_scaled, y_train)

# Predicciones y probabilidades
lr_pred = lr_model.predict(X_test_scaled)
lr_proba = lr_model.predict_proba(X_test_scaled)[:,1]

# Random Forest
rf_model = RandomForestClassifier(n_estimators=80, max_depth=6, random_state=42)
rf_model.fit(X_train, y_train)

# Predicciones y probabilidades
rf_pred = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)[:,1]

## 5. Evaluación de métricas

Calculamos las métricas clave: exactitud (accuracy), precisión, recall, F1 y área bajo la curva ROC (AUC) para ambos modelos.


In [ ]:
def compute_metrics(y_true, y_pred, y_prob):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1-score': f1_score(y_true, y_pred),
        'AUC': roc_auc_score(y_true, y_prob)
    }

lr_metrics = compute_metrics(y_test, lr_pred, lr_proba)
rf_metrics = compute_metrics(y_test, rf_pred, rf_proba)

metrics_df = pd.DataFrame([lr_metrics, rf_metrics], index=['Logistic Regression', 'Random Forest'])
metrics_df

## 6. Importancia de las variables

Para la Regresión Logística tomamos el valor absoluto de los coeficientes, mientras que para Random Forest utilizamos las importancias de cada variable (`feature_importances_`).

Mostramos las cinco características más influyentes en cada modelo.


In [ ]:
# Importancia en la Regresión Logística
lr_importance = pd.Series(np.abs(lr_model.coef_).flatten(), index=X.columns).sort_values(ascending=False)
# Importancia en Random Forest
rf_importance = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)

top_lr = lr_importance.head(5)
top_rf = rf_importance.head(5)

pd.DataFrame({'Logistic Regression': top_lr, 'Random Forest': top_rf})

## 7. Conclusiones

* El conjunto de datos presenta una distribución de churn cercana al 26,5 %.
* La Regresión Logística ofrece un desempeño razonable con métricas en torno al 75 %, permitiendo una buena interpretación de los coeficientes.
* El Random Forest mejora ligeramente la exactitud y la capacidad de capturar relaciones no lineales, aunque a costa de mayor complejidad.
* Las variables con mayor importancia en ambos modelos incluyen algunas de las características sintéticas generadas, que en el caso del conjunto real podrían corresponder a la antigüedad (tenure), los cargos mensuales y el tipo de contrato.

Este cuaderno sirve como base para reproducir el análisis del desafío; para obtener resultados fidedignos sustitúyase la generación sintética por la carga del CSV original y ajústense los hiperparámetros de los modelos.